In [ ]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns


# 设置随机种子
seed = 42  # 你可以选择任何整数作为随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # 如果使用的是 GPU，需要控制所有 GPU 的随机种子

# 设置设备（GPU / CPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 数据加载与预处理
file_path = 'concatenated_result917_8_9 - 2.csv'  # 修改为 CSV 文件路径
orig_data = pd.read_csv(file_path, encoding='ISO-8859-1')  # 使用 CSV 读取，并处理编码

# 计算特征和标签的相关性
corr_matrix = orig_data.corr()

# 画出热力图
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, cbar=True)
plt.title("特征与标签的皮尔逊相关系数热力图")
plt.show()

# 数据划分
train_num = int(len(orig_data) * 0.8)  # 80% 数据用于训练
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X = scaler_x.fit_transform(orig_data.iloc[:, :-1])
y = scaler_y.fit_transform(orig_data.iloc[:, -1].values.reshape(-1, 1))

# 制作时间序列数据集
def create_sequences(data_x, data_y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(data_x) - seq_len):
        X_seq.append(data_x[i:i + seq_len])
        y_seq.append(data_y[i + seq_len])
    return np.array(X_seq), np.array(y_seq)

seq_len = 30  # 固定滑动窗口长度
X_seq, y_seq = create_sequences(X, y, seq_len)

# 划分训练集和测试集
X_train, X_test = X_seq[:train_num], X_seq[train_num:]
y_train, y_test = y_seq[:train_num], y_seq[train_num:]

# 确定输入维度
input_dim = X_train.shape[2]

# 转换为 PyTorch 数据格式
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 定义 CNN 模型
class CNNModel(nn.Module):
    def __init__(self, input_dim, output_dim, seq_len, dropout):
        super(CNNModel, self).__init__()

        # 卷积层
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=128, kernel_size=3, stride=1, padding=1)

        # 全连接层
        self.fc = nn.Linear(128 * seq_len, output_dim)

        # Dropout层
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 卷积操作
        x = x.permute(0, 2, 1)  # 转换为 (batch_size, input_dim, seq_len) 形式
        x = self.conv1(x)
        x = self.conv2(x)

        # 展平
        x = x.view(x.size(0), -1)

        # Dropout
        x = self.dropout(x)

        # 全连接层
        output = self.fc(x)
        
        return output

# 初始化 CNN 模型
model = CNNModel(
    input_dim=input_dim,
    output_dim=1,
    seq_len=seq_len,
    dropout=0.2
).to(device)

# 打印模型架构（展示模型结构）
print(model)

# 损失函数与优化器
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-8)

# 训练模型
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs):
    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_losses.append(train_loss / len(train_loader))

        model.eval()
        test_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item()

        test_losses.append(test_loss / len(test_loader))

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}")

    return train_losses, test_losses

# 调用训练
train_losses, test_losses = train_model(
    model, train_loader, test_loader, criterion, optimizer, 400
)

# 可视化损失
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train and Test Loss Over Epochs")
plt.legend()
plt.show()

# 计算预测值和真实值
model.eval()
with torch.no_grad():
    # 只传递X_train或者X_test，作为CNN的输入
    y_train_pred = model(torch.tensor(X_train, dtype=torch.float32).to(device)).cpu().numpy()
    y_test_pred = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

y_train_true = scaler_y.inverse_transform(y_train)
y_train_pred_true = scaler_y.inverse_transform(y_train_pred)
y_test_true = scaler_y.inverse_transform(y_test)
y_test_pred_true = scaler_y.inverse_transform(y_test_pred)

# 手动实现 MAPE（如果 sklearn 版本不支持）
def mean_absolute_percentage_error(y_true, y_pred):
    epsilon = 1e-10  # 防止除以零
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon)) * 100)


# 统计指标
train_mse = mean_squared_error(y_train_true, y_train_pred_true)
test_mse = mean_squared_error(y_test_true, y_test_pred_true)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
train_mae = mean_absolute_error(y_train_true, y_train_pred_true)
test_mae = mean_absolute_error(y_test_true, y_test_pred_true)
train_mape = mean_absolute_percentage_error(y_train_true, y_train_pred_true)
test_mape = mean_absolute_percentage_error(y_test_true, y_test_pred_true)
train_r2 = r2_score(y_train_true, y_train_pred_true)
test_r2 = r2_score(y_test_true, y_test_pred_true)

# 格式化输出
print("\n=============== 模型评价指标 ===============")
print(f"训练集 MSE: {train_mse:.4f} \t 测试集 MSE: {test_mse:.4f}")
print(f"训练集 RMSE: {train_rmse:.4f} \t 测试集 RMSE: {test_rmse:.4f}")
print(f"训练集 MAE: {train_mae:.4f} \t 测试集 MAE: {test_mae:.4f}")
print(f"训练集 MAPE: {train_mape:.2f}% \t 测试集 MAPE: {test_mape:.2f}%")
print(f"训练集 R²: {train_r2:.4f} \t 测试集 R²: {test_r2:.4f}")
print("============================================")


In [ ]:
# 计算验证集和测试集的预测误差（绝对误差）
train_error = abs(y_train_pred_true - y_train_true)
test_error = abs(y_test_pred_true - y_test_true)

# 绘制绝对误差图
plt.figure(figsize=(12, 6))

# 训练集 - 绝对误差散点图
plt.scatter(range(len(train_error)), train_error, label='Train Absolute Error', color='blue', alpha=0.6, s=20)

# 测试集 - 绝对误差散点图
plt.scatter(range(len(train_error), len(train_error) + len(test_error)), test_error, label='Test Absolute Error', color='orange', alpha=0.6, s=20)

# 添加图例和标签
plt.xlabel('Index', fontsize=12, fontweight='bold')  # 使用索引作为横轴
plt.ylabel('Absolute Error', fontsize=12, fontweight='bold')  # 绝对误差作为 Y 轴
plt.legend(loc='best', fontsize=10, frameon=True, fancybox=True, framealpha=0.5)
plt.title('Train and Test Absolute Error (Scatter)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 显示图像
plt.show()


In [ ]:
# 计算验证集和测试集的预测误差（绝对误差）
train_error = abs(y_train_pred_true - y_train_true)
test_error =abs(y_test_pred_true - y_test_true)

# 计算相对误差
train_relative_error = 100 * train_error / y_train_true
test_relative_error = 100 * test_error / y_test_true

# 绘制相对误差图
plt.figure(figsize=(12, 6))

# 训练集 - 相对误差散点图
plt.scatter(range(len(train_relative_error)), train_relative_error, label='Train Relative Error', color='blue', alpha=0.6, s=20)

# 测试集 - 相对误差散点图
plt.scatter(range(len(train_relative_error), len(train_relative_error) + len(test_relative_error)), test_relative_error, label='Test Relative Error', color='orange', alpha=0.6, s=20)

# # 添加水平线 y=0
# plt.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero Relative Error')

# 添加图例和标签
plt.xlabel('Index', fontsize=12, fontweight='bold')  # 使用索引作为横轴
plt.ylabel('Relative Error (%)', fontsize=12, fontweight='bold')  # 相对误差作为 Y 轴
plt.legend(loc='best', fontsize=10, frameon=True, fancybox=True, framealpha=0.5)
plt.title('Train and Test Relative Error (Scatter)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 显示图像
plt.show()


In [ ]:
# 绘制训练集和测试集的散点图，真实值 vs 预测值
plt.figure(figsize=(12, 6))

# 训练集 - 真实值 vs 预测值
plt.scatter(range(len(y_train_true)), y_train_true, label='True Train', color='blue', alpha=0.6, s=30)
plt.scatter(range(len(y_train_pred_true)), y_train_pred_true, label='Predicted Train', color='green', alpha=0.6, s=30)

# 测试集 - 真实值 vs 预测值
plt.scatter(range(len(y_train_true), len(y_train_true) + len(y_test_true)), y_test_true, label='True Test', color='orange', alpha=0.6, s=30)
plt.scatter(range(len(y_train_true), len(y_train_true) + len(y_test_pred_true)), y_test_pred_true, label='Predicted Test', color='red', alpha=0.6, s=30)

# 添加图例和标签
plt.xlabel('Index', fontsize=12, fontweight='bold')  # 使用索引作为横轴
plt.ylabel('Value', fontsize=12, fontweight='bold')  # SOH 或其他值作为 Y 轴
plt.legend(loc='best', fontsize=10, frameon=True, fancybox=True, framealpha=0.5)
plt.title('True vs Predicted Values (Train and Test) - Scatter Plot', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 显示图像
plt.show()


In [ ]:
# 将预测值和真实值保存到 Excel 中
train_results = pd.DataFrame({
    'True Values': y_train_true.flatten(),
    'Predicted Values': y_train_pred_true.flatten(),
    'Train Error': train_error.flatten(),
    'Train Relative Error (%)': train_relative_error.flatten()
})

test_results = pd.DataFrame({
    'True Values': y_test_true.flatten(),
    'Predicted Values': y_test_pred_true.flatten(),
    'Test Error': test_error.flatten(),
    'Test Relative Error (%)': test_relative_error.flatten()
})

# 保存到 Excel
train_results.to_excel('train_results_dataset_2-CNN2.xlsx', index=False)
test_results.to_excel('test_results_dataset_2-CNN2.xlsx', index=False)

# 对第二个数据集做同样的操作
# 注意，你需要在训练第二个数据集后，重复相同的操作来保存第二个数据集的结果
